# Attention-zeroed post-layer-1 trajectory

Zero the layer-1 attention output while retaining the pre-layer-1 residual and the layer-1 MLP. This isolates the residual-plus-MLP condition from the clean post-layer-1 trajectory.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6010
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))

Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6010 line width 61


In [2]:
ATTN_OUT,POST="blocks.1.hook_attn_out","blocks.1.hook_resid_post"
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=[POST])
clean=c[POST][0].detach().cpu().numpy()
captured={}
def zero_attention(x,hook): return torch.zeros_like(x)
def capture(x,hook): captured["post"]=x.detach().cpu(); return x
model.run_with_hooks(batch.tokens,prepend_bos=False,fwd_hooks=[(ATTN_OUT,zero_attention),(POST,capture)])
attention_zeroed=captured["post"][0].numpy()
print("post-layer-1 MSE",float(np.mean((attention_zeroed-clean)**2)))
coords=shared_pca([clean,attention_zeroed])
plot_trajectory(coords,["clean post layer 1","attention output zeroed"],"Post-layer-1 trajectory with attention removed","figures/layer1_attention_zeroed.png")

post-layer-1 MSE 8.778180122375488
